# 🚀 Gradio Web App — News Topic Classifier
Launch an interactive web interface to classify news headlines in real time.

> Run all cells top-to-bottom.  
> A public shareable link is generated in the last cell.


## 1 · Install & Imports

In [8]:
!pip install gradio transformers torch -q

In [9]:
import numpy as np
import torch
import gradio as gr
from transformers import BertTokenizerFast, BertForSequenceClassification

MODEL_DIR   = "models/bert_ag_news"
MAX_LENGTH  = 128
LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
EMOJI = {"World": "🌍", "Sports": "⚽", "Business": "💼", "Sci/Tech": "🔬"}


## 2 · Load Model

In [10]:
from pathlib import Path

model_path = MODEL_DIR if Path(MODEL_DIR).exists() else "bert-base-uncased"
if model_path != MODEL_DIR:
    print("⚠️  Fine-tuned model not found — using base BERT (untrained).")
    print("   Run the training notebook first for proper results.")

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizerFast.from_pretrained(model_path)
model     = BertForSequenceClassification.from_pretrained(
    model_path, num_labels=4,
    id2label=LABEL_NAMES,
    label2id={v: k for k, v in LABEL_NAMES.items()},
    ignore_mismatched_sizes=True,
)
model.eval().to(device)
print(f"Model loaded on {device} ✅")


OSError: Error no file named model.safetensors, or pytorch_model.bin, found in directory models/bert_ag_news.

## 3 · Inference Function

In [ ]:
def classify(headline: str):
    if not headline.strip():
        return "⚠️ Please enter a headline.", {}

    inputs = tokenizer(
        headline, return_tensors="pt",
        truncation=True, max_length=MAX_LENGTH, padding=True,
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    idx   = int(np.argmax(probs))
    label = LABEL_NAMES[idx]
    conf  = float(probs[idx]) * 100

    result_md = f"### {EMOJI[label]} {label}\n**Confidence: {conf:.1f}%**"
    bar_data  = {f"{EMOJI[LABEL_NAMES[i]]} {LABEL_NAMES[i]}": float(probs[i])
                 for i in range(4)}
    return result_md, bar_data

# Quick test
r, b = classify("Scientists discover a new exoplanet with potential for life")
print(r)
print(b)


## 4 · Build Gradio UI

In [ ]:
EXAMPLES = [
    ["WHO warns of new respiratory disease outbreak in Southeast Asia"],
    ["Manchester City defeats Real Madrid 3-1 in Champions League final"],
    ["Federal Reserve raises interest rates by 25 basis points"],
    ["NASA discovers water ice deposits near the lunar south pole"],
    ["Apple reports record revenue beating Wall Street expectations"],
    ["G20 leaders meet to discuss global climate commitments"],
]

CSS = """
#title    { text-align: center; }
#subtitle { text-align: center; color: #6B7280; }
footer    { display: none !important; }
"""

with gr.Blocks(
    title="News Classifier",
    theme=gr.themes.Soft(
        primary_hue="blue",
        font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
    ),
    css=CSS,
) as demo:
    gr.Markdown("# 📰 News Topic Classifier", elem_id="title")
    gr.Markdown(
        "Fine-tuned **BERT** on AG News · "
        "**World · Sports · Business · Sci/Tech**",
        elem_id="subtitle",
    )

    with gr.Row():
        with gr.Column(scale=3):
            txt_input = gr.Textbox(
                label="News Headline",
                placeholder="Type or paste a headline here …",
                lines=3,
            )
            with gr.Row():
                btn_classify = gr.Button("🔍 Classify", variant="primary", scale=2)
                btn_clear    = gr.Button("🗑️ Clear",    variant="secondary", scale=1)

        with gr.Column(scale=2):
            out_label = gr.Markdown(value="")
            out_conf  = gr.Label(label="Confidence Scores", num_top_classes=4)

    gr.Markdown("### 💡 Try an example")
    gr.Examples(examples=EXAMPLES, inputs=[txt_input], examples_per_page=6)

    with gr.Accordion("ℹ️ Model details", open=False):
        gr.Markdown("""
| Property | Value |
|---|---|
| Base model | `bert-base-uncased` |
| Dataset | AG News (120k train / 7.6k test) |
| Max tokens | 128 |
| Expected accuracy | ~94% |
        """)

    btn_classify.click(classify, [txt_input], [out_label, out_conf])
    txt_input.submit(classify,   [txt_input], [out_label, out_conf])
    btn_clear.click(lambda: ("", {}, ""), [], [txt_input, out_conf, out_label])

print("UI built ✅")


## 5 · Launch the App

In [ ]:
# share=True  → generates a public link valid for 72 h (great for Colab)
# share=False → local only (http://localhost:7860)

demo.launch(share=True, show_error=True)
